In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage import data, img_as_float
import imageio as iio
from scipy.ndimage import gaussian_laplace, gaussian_filter,maximum_filter
from matplotlib.collections import PatchCollection
import cv2 as cv

In [ ]:
image=img_as_float(data.coins())
#print(image.dtype)
print(image.shape)
#Normalized LoG

def detectblobsa(img,sigmas,t=0.01):
    volume=np.zeros((len(sigmas), img.shape[0],img.shape[1]))
    for i,sig in enumerate(sigmas):
        volume[i]=(sig**2)*gaussian_laplace(img,sigma=sig)

    #Find 3D local maxima in scale space
    max_filter=maximum_filter(volume,size=3)
    is_max=(volume==max_filter)&(volume>t)
    coords=np.argwhere(is_max)
    blobs=[]
    for c in coords:
        sig_idx,y,x=c
        blobs.append((sigmas[sig_idx],y,x))
    return(blobs)

sigmas_log = np.logspace(np.log10(2), np.log10(15), num=12)
#tr=5
log_blobs = detectblobsa(image, sigmas_log, t=0.01)
fig, ax = plt.subplots(1,2,figsize=(12, 6))
ax[0].imshow(image,cmap='gray')
ax[1].imshow(image,cmap='gray')
for sigma,y,x in log_blobs:
    radius=np.sqrt(2)*sigma
    circle=Circle((x,y),radius,color='red',linewidth=2.0,fill=False)
    ax[1].add_patch(circle)

plt.show()

In [ ]:
def detectblobsb(img,sigmas,t=0.005):
    g_images=[gaussian_filter(img,sigma=sig) for sig in sigmas]
    dog_volume=np.zeros((len(sigmas)-1,img.shape[0],img.shape[1]))
    for i in range(len(sigmas)-1):
        dog_volume[i]=g_images[i+1]-g_images[i]

    #Find Maxima
    max_filtered=maximum_filter(dog_volume,size=3)
    is_max=(dog_volume==max_filtered)&(np.abs(dog_volume)>t)
    coords=np.argwhere(is_max)
    blobs=[]
    for c in coords:
        sig_idx,y,x=c
        avg_sigma=(sigmas[sig_idx]+sigmas[sig_idx+1])/2.0
        blobs.append((avg_sigma,y,x))
    return(blobs)

sigmas_log = np.logspace(np.log10(2), np.log10(15), num=12)
#tr=2
#print(detectblobsb(image, sigmas_log, t=0.01))
log_blobs = detectblobsb(image, sigmas_log, t=0.005)
fig, ax = plt.subplots(1,2,figsize=(12, 6))
ax[0].imshow(image,cmap='gray')
ax[1].imshow(image,cmap='gray')
patches=[]
for sigma,y,x in log_blobs:

    radius=np.sqrt(2)*sigma
    circle=Circle((x,y),radius,color='red',linewidth=2.0,fill=False)
    patches.append(circle)
p = PatchCollection(patches,match_original=True)
ax[1].add_collection(p)

plt.show()

In [ ]:
def detectblobDog(img,sigmas,t=0.005):
    g_images=[gaussian_filter(img,sigma=sig) for sig in sigmas]
    volume=np.zeros((len(sigmas-1),img.shape[0],img.shape[1]))
    for i in range(len(sigmas)-1):
        volume[i]=g_images[i+1]-g_images[i]
    max_filtered=maximum_filter(volume,size=3)
    is_max=(volume==max_filtered)&(np.abs(volume)>t)
    coords=np.argwhere(is_max)
    blobs=[]
    for c in coords:
        sig_idx,y,x=c
        avg_sigma=(sigmas[sig_idx]+sigmas[sig_idx+1])/2
        blobs.append((y,x,avg_sigma))
    return(blobs)

sigmas_log=np.logspace(np.log10(2),np.log10(15),num=12)
blobs=detectblobDog(image,sigmas_log,t=0.005)
print("Detected blobs:", len(blobs))

In [7]:
def assign_orientation(img, x, y, sigma):

    smooth = gaussian_filter(img, sigma=sigma)

    gy, gx = np.gradient(smooth)

    mag = np.sqrt(gx**2 + gy**2)
    angle = np.arctan2(gy, gx) % (2*np.pi)

    radius = int(np.ceil(4.5*sigma))

    x0, y0 = int(round(x)), int(round(y))

    if (x0-radius < 0 or x0+radius >= img.shape[1]
        or y0-radius < 0 or y0+radius >= img.shape[0]):
        return None

    yy, xx = np.mgrid[
        y0-radius:y0+radius+1,
        x0-radius:x0+radius+1
    ]

    weights = np.exp(
        -((xx-x)**2 + (yy-y)**2) /
        (2*(1.5*sigma)**2)
    )

    local_angle = angle[
        y0-radius:y0+radius+1,
        x0-radius:x0+radius+1
    ]

    local_mag = mag[
        y0-radius:y0+radius+1,
        x0-radius:x0+radius+1
    ]

    # 36-bin orientation histogram
    hist = np.zeros(36)

    bin_index = (
        np.floor(local_angle*36/(2*np.pi)).astype(int)
    ) % 36

    np.add.at(hist, bin_index, local_mag*weights)

    # Dominant orientation
    k = np.argmax(hist)

    # Parabolic interpolation
    left = hist[(k-1) % 36]
    centre = hist[k]
    right = hist[(k+1) % 36]

    denominator = left - 2*centre + right

    delta = (
        0.5*(left-right)/denominator
        if abs(denominator) > 1e-10 else 0
    )

    theta = (k+delta)*2*np.pi/36

    return theta


In [8]:
def calculate_descriptor(img, x, y, sigma, theta):

    d = 4
    n = 8

    hist_width = 3*sigma

    smooth = gaussian_filter(img, sigma=sigma)

    gy, gx = np.gradient(smooth)

    mag = np.sqrt(gx**2 + gy**2)
    angle = np.arctan2(gy, gx) % (2*np.pi)

    radius = int(np.ceil(
        hist_width*np.sqrt(2)*(d+1)/2
    ))

    x0, y0 = int(round(x)), int(round(y))

    if (x0-radius < 0 or x0+radius >= img.shape[1]
        or y0-radius < 0 or y0+radius >= img.shape[0]):
        return None

    # 4x4 cells, 8 orientation bins
    hist = np.zeros((d, d, n))

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    for py in range(y0-radius, y0+radius+1):

        for px in range(x0-radius, x0+radius+1):

            dx = px-x
            dy = py-y

            # Rotate coordinates by -theta
            xr = cos_t*dx + sin_t*dy
            yr = -sin_t*dx + cos_t*dy

            # Identify spatial cell
            bx = int(np.floor(xr/hist_width + d/2))
            by = int(np.floor(yr/hist_width + d/2))

            if not (0 <= bx < d and 0 <= by < d):
                continue

            # Relative orientation
            bo = (
                (angle[py, px]-theta) % (2*np.pi)
            )

            # Identify orientation bin
            bo = int(np.floor(bo*n/(2*np.pi))) % n

            # Gaussian weighting
            weight = np.exp(
                -(xr**2 + yr**2) /
                (2*(0.5*d*hist_width)**2)
            )

            # Hard assignment: one cell, one bin
            hist[by, bx, bo] += mag[py, px]*weight

    # Flatten: 4 x 4 x 8 = 128
    descriptor = hist.flatten()

    # First L2 normalization
    norm = np.linalg.norm(descriptor)

    if norm < 1e-10:
        return None, hist

    descriptor /= norm

    # Clipping
    descriptor = np.minimum(descriptor, 0.2)

    # Second L2 normalization
    descriptor /= np.linalg.norm(descriptor)

    return descriptor, hist

In [10]:
descriptors = []
keypoints = []
histograms = []

for y, x, sigma in blobs:

    theta = assign_orientation(image, x, y, sigma)

    if theta is None:
        continue

    result = calculate_descriptor(
        image, x, y, sigma, theta
    )

    if result is None:
        continue

    descriptor, hist = result

    if descriptor is None:
        continue

    descriptors.append(descriptor)
    histograms.append(hist)
    keypoints.append((x, y, sigma, theta))

descriptors = np.array(descriptors)

In [ ]:
index = 0
hist = histograms[index]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))

angles = np.arange(8)*2*np.pi/8

max_value = hist.max()

for i in range(4):
    for j in range(4):

        ax = axes[i, j]

        values = hist[i, j]

        if max_value > 0:
            values = values/max_value

        for k in range(8):

            length = values[k]

            ex = length*np.cos(angles[k])
            ey = length*np.sin(angles[k])

            ax.plot([0, ex], [0, ey], linewidth=2)

        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_aspect('equal')
        ax.set_title(f"Cell ({i+1},{j+1})")
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Simplified SIFT Descriptor: 4x4x8")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 3))

plt.imshow(
    descriptors[index].reshape(1, 128),
    cmap='gray',
    aspect='auto'
)

plt.title("128-Dimensional Descriptor")
plt.xlabel("Descriptor dimensions")
plt.yticks([])
plt.colorbar()
plt.show()